In [ ]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt 

In [ ]:
df = pd.read_csv(r'./data/insurance.csv')
df

### EDA

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
numeric_col = ['age','bmi','children','charges']
figure,axes = plt.subplots(1,4,figsize=(15,4))
for ax, col in zip(axes,numeric_col):
    sns.histplot(df[col],kde=True,bins=20,ax=ax)
    ax.set_title(f'Distribution of {col}',fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
categorial_col = ['sex','children','smoker','region']

figure, axes = plt.subplots(1,4,figsize=(15,4))

for ax, col in zip(axes,categorial_col):
    sns.countplot(x = df[col],ax=ax)
    ax.set_title(f'Distribution of {col}',fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
figure,axes = plt.subplots(1,4,figsize=(15,4))
for ax, col in zip(axes,numeric_col):
    sns.boxplot(x = df[col],ax=ax)
    ax.set_title(f'spreads of {col}',fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(df.corr(numeric_only=True),annot=True)
plt.show()

### Data Cleaning and preprocessing

In [ ]:
df_cleaned = df.copy()
df_cleaned.head()

In [ ]:
df_cleaned.shape

In [ ]:
df_cleaned.drop_duplicates(inplace=True)
df_cleaned.shape

In [ ]:
df_cleaned.isna().sum()

In [ ]:
df_cleaned.dtypes

In [ ]:
df_cleaned['sex'].value_counts()

In [ ]:
df_cleaned['sex'] = df_cleaned['sex'].map({'male':0,'female':1})
df_cleaned.head()

In [ ]:
df_cleaned['smoker'] = df_cleaned['smoker'].map({'no':0,'yes':1})
df_cleaned.head()

In [ ]:
df_cleaned.rename(columns={
    'sex' : 'is_female',
    'smoker' : 'is_smoker'
},inplace=True)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned['region'].value_counts()

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned,columns=['region'],drop_first=False,dtype=int)
df_cleaned.head()

### Feature Engineering and Extraction

In [ ]:
df_cleaned['bmi_category'] = pd.cut(
    df_cleaned['bmi'],
    bins=[0,18.5,24.9,29.9,float('inf')],
    labels=['Underweight','Normal','Overweight','Obese']
)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned,columns=['bmi_category'],drop_first=False,dtype=int)
df_cleaned.head()

In [ ]:
df_cleaned.columns

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df_cleaned.corr(),annot=True)

In [ ]:
from scipy.stats import pearsonr

# ----------------------------------
# Pearson Correlation Calculation
# ----------------------------------

# List of features to check against target
selected_features = [
    'age', 'bmi', 'children', 'is_female', 'is_smoker','region_northeast',
    'region_northwest', 'region_southeast', 'region_southwest','bmi_category_Underweight',
    'bmi_category_Normal', 'bmi_category_Overweight', 'bmi_category_Obese'
]

correlations = {
    feature: pearsonr(df_cleaned[feature], df_cleaned['charges'])[0]
    for feature in selected_features
}
correlation_df = pd.DataFrame(list(correlations.items()), columns=['Feature', 'Pearson Correlation'])
correlation_df.sort_values(by='Pearson Correlation', ascending=False)


In [ ]:
cat_features = [
    'is_female', 'is_smoker','region_northeast',
    'region_northwest', 'region_southeast', 'region_southwest','bmi_category_Underweight',
    'bmi_category_Normal', 'bmi_category_Overweight', 'bmi_category_Obese'
]

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

alpha = 0.05

df_cleaned['charges_bin'] = pd.qcut(df_cleaned['charges'], q=4, labels=False)
chi2_results = {}

for col in cat_features:
    contingency = pd.crosstab(df_cleaned[col], df_cleaned['charges_bin'])
    chi2_stat, p_val, _, _ = chi2_contingency(contingency)
    decision = 'Reject Null (Keep Feature)' if p_val < alpha else 'Accept Null (Drop Feature)'
    chi2_results[col] = {
        'chi2_statistic': chi2_stat,
        'p_value': p_val,
        'Decision': decision
    }

chi2_df = pd.DataFrame(chi2_results).T
chi2_df = chi2_df.sort_values(by='p_value')
chi2_df

In [ ]:
final_df = df_cleaned[['age', 'is_female', 'bmi', 'children', 'is_smoker', 'charges','region_southeast','bmi_category_Obese']]
final_df